<b>Fine-tuning Flair NER models</b>

This notebook demonstrates how to fine-tune Flair models using stacked contextual and word embeddings on a general-domain Portuguese dataset available at: https://doi.org/10.6084/m9.figshare.28970633.

<b>Import the required libraries</b>

In [ ]:
from flair.data import Corpus
from flair.datasets import ColumnCorpus
from flair.models import SequenceTagger
from flair.embeddings import FlairEmbeddings, StackedEmbeddings, TokenEmbeddings, WordEmbeddings, TransformerWordEmbeddings
from flair.trainers import ModelTrainer
from typing import List

<b>Load ground-truth files</b>

In [ ]:
# Dataset options https://doi.org/10.6084/m9.figshare.28970633
src_path="ner/datasets/ner-domain-gen/"

columns = {0: 'text', 1: 'ner'}
corpus: Corpus = ColumnCorpus(src_path, columns, train_file='train.tsv', test_file='test.tsv', in_memory=True)
tag_dictionary = corpus.make_label_dictionary(label_type="ner")

<b>Load language models</b>

In [ ]:
# Skip-gram Word2Vec model options at http://nilc.icmc.usp.br/nilc/index.php/repositorio-de-word-embeddings-do-nilc
word2vec = WordEmbeddings('path/to/word2vec/models/skip_s300.txt')

# FlairEL embedding model options at https://github.com/ericlief/language_models
# FlairBBP embedding model options at https://github.com/jneto04/ner-pt
flair_embedding_forward = FlairEmbeddings('path/to/flair/embeddings/flairBBP_forward-pt.pt')
flair_embedding_backward = FlairEmbeddings('path/to/flair/embeddings/flairBBP_backward-pt.pt')

embedding_types: List[TokenEmbeddings] = [word2vec, flair_embedding_forward, flair_embedding_backward]
embeddings: StackedEmbeddings = StackedEmbeddings(embeddings=embedding_types)

# Load sequence labeling model with default hyperparameters
model = SequenceTagger(hidden_size=256, embeddings=embeddings, tag_dictionary=tag_dictionary, tag_type="ner", use_crf=True)

<b>Fine-tune NER model with embeddings stack</b>

In [ ]:
trainer: ModelTrainer = ModelTrainer(model, corpus)

trainer.train('path/to/your/output/test.tsv',
              learning_rate=0.1,
              mini_batch_size=32,
              embeddings_storage_mode='gpu',
              max_epochs=150,
              checkpoint=True,
              optimizer=torch.optim.SGD,
)